# 01. Capacitaciones (todas)

**Fuente:** `data/raw/capacitacionestodas.csv`  
**Salida:** `data/processed/capacitaciones_todas.csv`

Cursos, seminarios y diplomados tomados por el personal de ESPOL. Es una de las fuentes de formacion continua para el perfil de cada persona.

In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd()))
import pandas as pd
import _preprocesamiento_comun as pc

pd.set_option('display.max_columns', 100)

## 1. Carga de datos crudos

In [ ]:
df = pc.leer_csv('capacitacionestodas.csv', low_memory=False)
df.head()

### Diccionario de valores (columnas descriptivas)

Estos campos vienen ya traducidos desde el origen (SQL), pero se documentan aquí los códigos/valores posibles para referencia:

- **TIPODESCRIPCION**: `APROBACION` o `ASISTENCIA`.
- **AREACAPACITACIONDESCRIPCION**: `DI` → "Disciplinar", `PE` → "Pedagógica", `OT` → "Otra".
- **TIPOMODALIDADDESCRIPCION** (derivado de `TIPOMODALIDAD`):
  ```sql
  CASE
      WHEN C.TIPOMODALIDAD = 'P' THEN 'PRESENCIAL'
      WHEN C.TIPOMODALIDAD = 'L' THEN 'VIRTUAL'
      ELSE ''
  END AS TIPOMODALIDADDESCRIPCION
  ```
- **FORMACONOCIMDESCRIPCION** (derivado de `FORMACONOCIM`):
  ```sql
  CASE C.FORMACONOCIM
      WHEN 'R' THEN 'PARTICIPANTES-RECIBIDOS'
      WHEN 'I' THEN 'INSTRUCTOR-IMPARTIDOS'
      WHEN 'A' THEN 'ASISTENCIA'
      WHEN 'P' THEN 'PRESENTACION'
      ELSE C.FORMACONOCIM
  END AS FORMACONOCIMDESCRIPCION
  ```

## 2. Exploración inicial

In [ ]:
pc.resumen(df, 'capacitaciones_todas')

In [ ]:
df.dtypes

## 3. Limpieza

In [ ]:
df = pc.limpiar_strings(df)
df = pc.quitar_columnas_vacias(df, umbral=0.86)
df = pc.quitar_columnas_constantes(df, excluir=["FORMACONOCIM"])

In [ ]:
# Columnas administrativas/de auditoría del sistema origen que no aportan al perfil.
# IDINSTITUCION se descarta porque, al explorar el crudo, el 99.99% de los valores
# está vacío o en 0 (solo 6 filas de 64563 tienen un id real) -> sin valor informativo.
# TIPOEVENTO, IDTIPOCONOCIMIEN y TIPOMODALIDAD son los códigos crudos de columnas que
# ya tienen su versión descriptiva (TIPOEVENTODESCRIPCION, TIPOMODALIDADDESCRIPCION);
# FORMACONOCIM se conserva igualmente aunque sea constante (se deja fijo por decisión
# explícita, no se elimina junto con las demás columnas constantes). ULTIMO_CAMBIO es
# metadato técnico de auditoría del sistema origen, no aporta al perfil. FECHAASCENSO
# corresponde al escalafón/ascenso de la persona, no a la capacitación en sí.
columnas_sin_aporte = [
    'GESTIONUATHESPOL', 'ORIGENINGRESO', 'INGRESORRHH', 'IDUSUARIO',
    'CONSIDERACOMISANT', 'ENESCALAFON', 'REVISADOPARAESCALAFON', 'IDPAIS',
    'TIPOCERTIFICADO', 'TIPOCAPACITACION', 'IDINSTITUCION',
    'TIPOEVENTO',  'NAMEARCHDOC', 'IDTIPOCONOCIMIEN', 'TIPOMODALIDAD',
    'ULTIMO_CAMBIO', 'FECHAASCENSO',
]
df = df.drop(columns=columnas_sin_aporte, errors='ignore')
print(f'Columnas eliminadas por no aportar al perfil: {columnas_sin_aporte}')

In [ ]:
# AREACAPACITACIONDESCRIPCION en el crudo trae el código, no la descripción.
# Al explorar sus valores únicos solo aparecen 'DI', 'PE', 'OT' (y vacíos) -> se mapean
# a su descripción según el diccionario documentado arriba.
mapeo_area_capacitacion = {'DI': 'Disciplinar', 'PE': 'Pedagógica', 'OT': 'Otra'}
valores_no_mapeados = set(df['AREACAPACITACIONDESCRIPCION'].dropna().unique()) - set(mapeo_area_capacitacion)
assert not valores_no_mapeados, f'Códigos sin mapeo en AREACAPACITACIONDESCRIPCION: {valores_no_mapeados}'
df['AREACAPACITACIONDESCRIPCION'] = df['AREACAPACITACIONDESCRIPCION'].map(mapeo_area_capacitacion)
df['AREACAPACITACIONDESCRIPCION'].value_counts(dropna=False)

In [ ]:
# FORMACONOCIM en el crudo trae la letra, no la descripción. Se mapea según el
# diccionario documentado arriba (FORMACONOCIMDESCRIPCION); los valores que no estén
# en el diccionario se dejan tal cual (ELSE C.FORMACONOCIM en el SQL de origen).
mapeo_forma_conocim = {
    'R': 'PARTICIPANTES-RECIBIDOS',
    'I': 'INSTRUCTOR-IMPARTIDOS',
    'A': 'ASISTENCIA',
    'P': 'PRESENTACION',
}
df['FORMACONOCIM'] = df['FORMACONOCIM'].replace(mapeo_forma_conocim)
df['FORMACONOCIM'].value_counts(dropna=False)

In [ ]:
pc.resumen(df)

## 5. Tipado de fechas e identificadores

In [ ]:
df = pc.castear_fechas(df, ['FECHAINICIO', 'FECHAFIN', 'FECHACHECKESCALAFON', 'FECHASUBIDAARCHIVO', 'FECHAAPROBACION'])
df = pc.castear_enteros(df, ['IDCAPACITACION', 'IDPERSONA', 'IDCIUDAD', 'IDCANTON'])

## 6. Gráficos exploratorios

Vistas rápidas para apoyar la construcción del catálogo de variables del perfil (Fase 1-2 de la metodología): estacionalidad/tendencia temporal, categorías dominantes y forma de la distribución de las variables numéricas.

In [ ]:
%matplotlib inline
import matplotlib.pyplot as plt
plt.rcParams['figure.figsize'] = (9, 4)

**Capacitaciones por año**: evolución de la actividad de formación continua del personal.

In [ ]:
pc.grafico_por_anio(df['FECHAINICIO'], 'Capacitaciones registradas por año')

**Tipos de evento más frecuentes**: cursos, seminarios, diplomados, etc.

In [ ]:
pc.grafico_barras(df['TIPOEVENTODESCRIPCION'], 'Top 10 tipos de evento', top=10)

**Capacitaciones por persona**: cuántas capacitaciones acumula cada persona; es la base de una futura variable de "nivel de formación continua" por perfil.

In [ ]:
conteo_por_persona = df['IDPERSONA'].value_counts()
pc.grafico_histograma(conteo_por_persona, 'Distribución del número de capacitaciones por persona', bins=30)

## 7. Verificación final

In [ ]:
pc.resumen(df, 'capacitaciones_todas (procesado)')
df.head()

## 8. Guardado en data/processed

In [ ]:
pc.guardar_procesado(df, 'capacitaciones_todas.csv')